# Data Ingestion and Cleaning

We load the reviews and metadata datasets for Health and Personal Care, clean them up and merge them into a single dataframe ready for the next steps.

In [ ]:
# Importing libraries
import pandas as pd
import numpy as np
import re
from html import unescape

In [ ]:
# ── Configuration — change only here, never inside functions ──────────────────
# Products
MIN_REVIEWS          = 10    # products with fewer reviews are excluded
CAP_PRODUCT_TOKENS   = 400   # token cap for product text (MPNet max ≈ 512 subword)
CAP_FEATURES_TOKENS  = 150   # max tokens from features
CAP_DESC_TOKENS      = 150   # max tokens from description
CAP_DETAILS_TOKENS   = 80    # max tokens from semantic details

# Reviews
MIN_REVIEW_TOKENS    = 25    # min tokens per review (removes "Love it!!", "Fast shipping", etc.)

# Balanced rating selection — reduces 5-star bias from ~70% raw → ~48%
REVIEWS_PER_RATING = {
    5: 5,   # max 5 five-star reviews (most helpful first)
    4: 3,   # max 3 four-star
    3: 2,   # max 2 three-star (neutral, low signal)
    2: 3,   # max 3 two-star
    1: 3,   # max 3 one-star (critical, important signal)
}

print("Configuration:")
print(f"  Products : min {MIN_REVIEWS} reviews, text cap {CAP_PRODUCT_TOKENS} tokens")
print(f"  Reviews  : min {MIN_REVIEW_TOKENS} tokens, balanced selection {REVIEWS_PER_RATING}")
print(f"  Max reviews/product: {sum(REVIEWS_PER_RATING.values())}")

In [ ]:
# Importing the two dataset provided by Deloitte
df_meta = pd.read_json("data/meta_Health_and_Personal_Care.json", lines=True) 
df_review = pd.read_json("data/Health_and_Personal_Care.json", lines=True)
# Printing the info about the two Dataset
print(f"✅ Metadata loaded: {df_meta.shape[0]:,} rows × {df_meta.shape[1]} columns")
print(f"✅ Reviews loaded:  {df_review.shape[0]:,} rows × {df_review.shape[1]} columns")

In [ ]:
# Checking the first rows of the two datasets
print("Metadata")
display(df_meta.head())
print("\nReviews")
display(df_review.head())

In [ ]:
# Checking the datatypes
print("Meta dtypes:\n", df_meta.dtypes)
print("\nReview dtypes:\n", df_review.dtypes)

In [ ]:
# Missing values BEFORE converting empty lists/dicts/strings to NaN
meta_missing_raw = df_meta.isna().mean().sort_values(ascending=False)
review_missing_raw = df_review.isna().mean().sort_values(ascending=False)

print("Missing meta (raw):\n", meta_missing_raw.head(15))
print("\nMissing review (raw):\n", review_missing_raw.head(15))

## Handling hidden missing values

Some columns have empty lists, empty dicts or empty strings that pandas doesn't see as NaN, so we convert them all.

In [ ]:
# Converting [], {}, and "" to NaN in both dataframes. Only object columns are checked 

for col in df_meta.select_dtypes(include="object").columns:
    df_meta.loc[df_meta[col].isin([[], {}, ""]), col] = np.nan

for col in df_review.select_dtypes(include="object").columns:
    df_review.loc[df_review[col].isin([[], {}, ""]), col] = np.nan

In [ ]:
# Counting missing values AFTER cleanup
meta_missing = df_meta.isna().mean().sort_values(ascending=False)
review_missing = df_review.isna().mean().sort_values(ascending=False)

print("Meta missing ratio (clean):\n", meta_missing.head(20))
print("\nReview missing ratio (clean):\n", review_missing.head(20))

# Drop columns that are 100% NaN 
cols_all_nan = [col for col in df_meta.columns if df_meta[col].isna().all()]
print(f"\nDropping 100% NaN columns: {cols_all_nan}")
df_meta.drop(columns=cols_all_nan, inplace=True)

## Filtering

We only keep verified purchases and products with more than 10 reviews to avoid spam and statistically meaningless data.

In [ ]:
# Keeping only verified purchases
print(f"Reviews before verified filter: {len(df_review)}")
df_review = df_review[df_review["verified_purchase"]].copy()
print(f"Reviews after verified filter:  {len(df_review)}")

In [ ]:
# Keeping only products with more than 10 reviews
MIN_REVIEWS = 10

review_counts = df_review.groupby("parent_asin").size()
valid_asins = review_counts[review_counts >= MIN_REVIEWS].index

print(f"Unique products before filter: {df_review['parent_asin'].nunique()}")
df_review = df_review[df_review["parent_asin"].isin(valid_asins)].copy()
print(f"Unique products after filter (>{MIN_REVIEWS} reviews): {df_review['parent_asin'].nunique()}")
print(f"Total reviews kept: {len(df_review)}")

## Text Cleaning

We remove HTML tags, special characters and extra whitespace from review and metadata text.

New helper functions added:
- **`clean_text`** — strips HTML, normalises apostrophes *and* unicode dashes (`–` `—` → `-`)
- **`flatten_to_text`** — converts list or string to clean text with optional token cap
- **`extract_details_semantic`** — selects the 18+ semantically useful keys from `details` (Skin Type, Material, Flavor, etc.) and separates FORM fields (Item Form, Dosage Form) which go at the top of the product text to disambiguate category immediately (shampoo vs capsule vs cream vs device)
- **`get_brand`** — extracts brand from multiple fallback sources: `brand` column, `store` column, `details` dict

In [ ]:
def clean_text(text):
    """Strip HTML tags, unescape HTML entities, normalise unicode chars, collapse whitespace."""
    if pd.isna(text) or not text:
        return text
    text = unescape(str(text))
    # Normalize smart/curly apostrophes to plain ASCII apostrophe
    text = text.replace('\u2019', "'").replace('\u2018', "'")
    # Normalize unicode dashes to ASCII hyphen
    text = text.replace('\u2013', '-').replace('\u2014', '-')
    text = re.sub(r"<[^>]+>", " ", text)            # strip HTML tags
    text = re.sub(r"[^\w\s.,!?'/\-]", " ", text)    # keep only safe chars
    text = re.sub(r"\s+", " ", text).strip()         # collapse whitespace
    return text if len(text) > 2 else np.nan

def flatten_to_text(val, cap_tokens=None):
    """Convert list or string to clean text, with optional token cap."""
    if isinstance(val, list):
        parts = [clean_text(str(x)) for x in val if x and str(x).strip()]
        parts = [p for p in parts if pd.notna(p) and p]
        text  = ' '.join(parts)
    elif isinstance(val, str):
        text = clean_text(val)
    else:
        return np.nan
    if not text or pd.isna(text):
        return np.nan
    if cap_tokens:
        text = ' '.join(text.split()[:cap_tokens])
    return text if text else np.nan

# Semantically meaningful keys from 'details' — excludes physical dims, dates, codes
SEMANTIC_DETAIL_KEYS = {
    'Item Form', 'Material', 'Material Feature', 'Special Feature',
    'Flavor', 'Dosage Form', 'Color', 'Size', 'Scent', 'Skin Type',
    'Active Ingredients', 'Specific Uses For Product', 'Product Benefits',
    'Compatible Devices', 'Power Source', 'Age Range (Description)',
    'Style', 'Recommended Uses For Product', 'Department', 'Occasion',
    'Target Audience', 'Hair Type', 'Skin Tone',
}

# Form keys go at the top of product text to immediately disambiguate category
FORM_KEYS = {'Item Form', 'Dosage Form', 'Material'}

def extract_details_semantic(details, cap_tokens=None):
    """
    Extract semantic fields from the details dict.
    Returns (form_text, rest_text): FORM fields are separated so they can be
    placed at the top of the product text for category disambiguation.
    """
    if not isinstance(details, dict):
        return np.nan, np.nan
    form_parts, rest_parts = [], []
    for k, v in details.items():
        if k not in SEMANTIC_DETAIL_KEYS:
            continue
        v_str = str(v).strip()
        if not v_str or v_str.lower() in ('none', 'n/a', '-', '—', ''):
            continue
        (form_parts if k in FORM_KEYS else rest_parts).append(f"{k}: {v_str}")
    form_text = '. '.join(form_parts) if form_parts else np.nan
    rest_text = '. '.join(rest_parts) if rest_parts else np.nan
    if cap_tokens and pd.notna(rest_text):
        rest_text = ' '.join(str(rest_text).split()[:cap_tokens])
    return form_text, rest_text

def get_brand(row):
    """Extract brand from multiple fallback sources: brand col, store col, details dict."""
    for field in ('brand', 'store'):
        val = row.get(field)
        if val and str(val).strip().lower() not in ('none', '', 'nan'):
            return clean_text(str(val))
    d = row.get('details')
    if isinstance(d, dict):
        for key in ('Brand', 'Manufacturer'):
            val = d.get(key)
            if val and str(val).strip().lower() not in ('none', '', 'nan'):
                return clean_text(str(val))
    return np.nan

# Apply to review columns
df_review["title"] = df_review["title"].apply(clean_text)
df_review["text"]  = df_review["text"].apply(clean_text)

df_review[["title", "text"]].head()

## Merge metadata and reviews

We pull title, brand and description from the metadata and join them with the reviews on parent_asin.

In [ ]:
# Build enriched metadata fields using the new helper functions
df_meta["brand"]             = df_meta.apply(lambda r: get_brand(r.to_dict()), axis=1)
df_meta["description_clean"] = df_meta["description"].apply(lambda x: flatten_to_text(x, cap_tokens=CAP_DESC_TOKENS))
df_meta["features_clean"]    = df_meta["features"].apply(lambda x: flatten_to_text(x, cap_tokens=CAP_FEATURES_TOKENS))

# Extract semantic details — form_text (Item Form/Dosage Form) goes at top of product text
form_details = df_meta["details"].apply(lambda x: extract_details_semantic(x, cap_tokens=CAP_DETAILS_TOKENS))
df_meta["form_text"]    = form_details.apply(lambda x: x[0])
df_meta["details_rest"] = form_details.apply(lambda x: x[1])

# Clean text in metadata fields
df_meta["title"]             = df_meta["title"].apply(clean_text)
df_meta["description_clean"] = df_meta["description_clean"].apply(
    lambda x: clean_text(x) if pd.notna(x) else x
)

print(f"Brand coverage:       {df_meta['brand'].notna().mean():.2%}")
print(f"Description coverage: {df_meta['description_clean'].notna().mean():.2%}")
print(f"Features coverage:    {df_meta['features_clean'].notna().mean():.2%}")
print(f"Form coverage:        {df_meta['form_text'].notna().mean():.2%}")
print(f"Details coverage:     {df_meta['details_rest'].notna().mean():.2%}")

In [ ]:
# Selecting relevant metadata columns and merge with reviews
meta_cols = df_meta[[
    "parent_asin", "title", "brand", "description_clean", "features_clean",
    "form_text", "details_rest", "average_rating", "rating_number", "price", "store"
]].copy()

# Renaming to avoid column name clashes after merge
meta_cols.rename(columns={
    "title": "product_title",
    "average_rating": "product_avg_rating",
    "rating_number": "product_rating_count"
}, inplace=True)

# Merging on parent_asin (left join: keep all filtered reviews, attach metadata)
df = df_review.merge(meta_cols, on="parent_asin", how="left")

print(f"Merged dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

## Product-Level Dataset

We build a product-level table (one row per parent_asin) from the merged dataframe, combining title, brand and description into a single text field ready for embedding in the next step.

In [ ]:
# Product-level table (1 row per parent_asin)
meta_unique_cols = [
    "parent_asin", "product_title", "brand", "description_clean", "features_clean",
    "form_text", "details_rest", "product_avg_rating", "product_rating_count", "price", "store"
]
meta_unique_cols = [c for c in meta_unique_cols if c in df.columns]

products_df = df[meta_unique_cols].drop_duplicates("parent_asin").copy()

def build_product_text(row):
    """
    Assembles product_text_base in the optimal order for embedding:
    TITLE → BRAND → FORM → FEATURES → DESCRIPTION → SPECS

    FORM (Item Form / Dosage Form) is placed right after TITLE so the model
    immediately knows the product category before reading anything else.
    This disambiguates e.g. 'Biotin Shampoo' vs 'Biotin Capsule' which
    otherwise share nearly identical titles and descriptions.
    """
    parts = []
    if pd.notna(row.get("product_title")):
        parts.append(f"TITLE: {row['product_title']}")
    if pd.notna(row.get("brand")):
        parts.append(f"BRAND: {row['brand']}")
    if pd.notna(row.get("form_text")):
        parts.append(f"FORM: {row['form_text']}")
    if pd.notna(row.get("features_clean")):
        parts.append(f"FEATURES: {row['features_clean']}")
    if pd.notna(row.get("description_clean")):
        parts.append(f"DESCRIPTION: {row['description_clean']}")
    if pd.notna(row.get("details_rest")):
        parts.append(f"SPECS: {row['details_rest']}")

    full_text = "\n".join(parts)
    # Cap at token limit (approximate by word count)
    words = full_text.split()
    if len(words) > CAP_PRODUCT_TOKENS:
        full_text = " ".join(words[:CAP_PRODUCT_TOKENS])
    return full_text

products_df["product_text_base"] = products_df.apply(build_product_text, axis=1)
products_df["text_tokens"] = products_df["product_text_base"].str.split().str.len()

print("Products:", products_df.shape)
print(f"\nToken count per product (product_text_base):")
print(f"  mean={products_df['text_tokens'].mean():.0f}  "
      f"median={products_df['text_tokens'].median():.0f}  "
      f"min={products_df['text_tokens'].min()}  max={products_df['text_tokens'].max()}")
print(f"  < 15 tokens (poor text):  {(products_df['text_tokens'] < 15).mean():.1%}")
print(f"  > 50 tokens (rich text):  {(products_df['text_tokens'] > 50).mean():.1%}")
products_df[["parent_asin", "product_text_base"]].head(3)

## Review Selection — Balanced by Rating

Instead of picking the top-N reviews by helpful votes alone (which yields ~70% 5-star), we apply a **balanced selection** strategy:

| Rating | Cap | Reason |
|--------|-----|--------|
| 5★ | 5 | Most common — capped to reduce bias |
| 4★ | 3 | Positive with nuance |
| 3★ | 2 | Neutral, low signal — small cap |
| 2★ | 3 | Negative signal |
| 1★ | 3 | Critical signal — important for summaries |

Within each band, reviews are ranked by `helpful_vote` descending.

Reviews shorter than `MIN_REVIEW_TOKENS` tokens are excluded first to remove noise ("Love it!!", "Fast shipping", etc.).

In [ ]:
review_cols = ["parent_asin", "title", "text", "rating", "verified_purchase", "helpful_vote", "timestamp"]
review_cols = [c for c in review_cols if c in df.columns]

reviews_df = df[review_cols].copy()
reviews_df["helpful_vote"] = reviews_df["helpful_vote"].fillna(0)
reviews_df["text"] = reviews_df["text"].fillna("").astype(str).str.strip()
reviews_df = reviews_df[reviews_df["text"].str.len() > 0]

# Filter out reviews that are too short — removes noise ("Love it!!", "Fast shipping", etc.)
reviews_df["tok_len"] = reviews_df["text"].str.split().str.len()
before_tok = len(reviews_df)
reviews_df = reviews_df[reviews_df["tok_len"] >= MIN_REVIEW_TOKENS].copy()
print(f"Reviews after min-token filter (>={MIN_REVIEW_TOKENS} tokens): {len(reviews_df):,}  "
      f"(removed {before_tok - len(reviews_df):,})")

# Balanced selection by rating — reduces 5-star bias from ~70% raw to ~48%
def select_balanced(grp):
    """Select reviews with balanced rating distribution, most helpful first within each band."""
    selected = []
    for rating, cap in REVIEWS_PER_RATING.items():
        bucket = grp[grp["rating"] == rating].nlargest(cap, "helpful_vote")
        selected.append(bucket)
    return pd.concat(selected) if selected else grp.head(0)

reviews_top_df = (
    reviews_df
    .groupby("parent_asin", group_keys=False)
    .apply(select_balanced)
    .reset_index(drop=True)
)

print(f"\nTop reviews (balanced selection): {reviews_top_df.shape}")
reviews_top_df.head(3)

In [ ]:
counts = reviews_top_df.groupby("parent_asin").size()
print(f"Reviews per product — min: {counts.min()}  max: {counts.max()}  mean: {counts.mean():.1f}")
print(f"Products with < 5 reviews selected: {(counts < 5).sum()}")

# Rating distribution after balancing
print("\nRating distribution (after balanced selection):")
total = len(reviews_top_df)
for r in [1, 2, 3, 4, 5]:
    n   = (reviews_top_df["rating"] == r).sum()
    bar = '█' * int(n / total * 40)
    print(f"  {r}★ {bar:<40} {n:,} ({n/total:.1%})")

print(f"\nToken stats (reviews):")
print(f"  mean={reviews_top_df['tok_len'].mean():.0f}  "
      f"median={reviews_top_df['tok_len'].median():.0f}  "
      f"p90={reviews_top_df['tok_len'].quantile(0.9):.0f}")

In [ ]:
# Saving the cleaned and merged dataset for the next steps
df.to_csv("data/reviews_cleaned.csv", index=False)
products_df.to_csv("data/products_cleaned.csv", index=False)
reviews_top_df.to_csv("data/reviews_topN.csv", index=False)

print("Saved: data/reviews_cleaned.csv")
print("Saved: data/products_cleaned.csv")
print("Saved: data/reviews_topN.csv")